# Instructions

1. Assign subject to a group using [this random number generator](https://www.calculator.net/random-number-generator.html?slower=1&supper=4&ctype=1&s=5113&x=Generate).
2. Enter the date, subject ID, and group info into [the table on the timeline slides](https://docs.google.com/presentation/d/1vPCzzxUqa3gjfo_nY4hteUsnp4uf-nPtYLgrwr6Yat8/edit#slide=id.g2fc55de8142_0_754)
3. Show the participant the [task explanation slides](https://docs.google.com/presentation/d/1gwbIJBaGXZgyK8oOKZCZh0Squ2SrUesE2eKkt39Vv1I/edit#slide=id.p). Reach directly off of the slides -- try not to add any additional instructions or explanations.
4. Have the participant sign the [consent form](https://redcap.iths.org/surveys/?s=FCXJ4HP7KEM3EJWW).
5. Make sure that chair legs are overlying tape markers on the floor, and seat the participant.
6. On the BMI3D computer, double check that the “master” branch of BMI3D is checked out in VScode (bottom left corner) and click the little refresh symbol to ensure you have the most recent version.
7. In Google Chrome (BMI3D computer), click the BMI3d bookmark in the bookmarks bar and click Setup. Click Subjects and add a new subject with the subject ID. Subject name should be the study abbreviation followed by the subject’s assigned number, i.e. DPS006
8. Measure from the floor to the participants’ eye level (in cm)
9. Put your first name (capitalized) as the experiment, the subject number, group number, and the participant's eye height in the variables below.

## Recording Info

In [ ]:
experimenter = "Victoria"
subject = "DPS000"
group = 1
eye_height = 130 # measure in cm

save = True # save data (false is test)




#### ---- Do not edit below here --- ####

if not experimenter[0].isupper():
    print("Warning: Capitalize your name for consistency. Check spelling.")
if subject[:3] != "DPS" or not subject[3:].isdigit() or len(subject) != 6:
    print(subject[:3] != "DPS", ~subject[3:].isdigit(), len(subject) != 6)
    print("Warning: subject names must be DPS followed by three-digit number.")
if group not in [1,2,3,4]:
    print("Warning: experimental condition not valid. Choose from:\n"
    "1: Prior-Aligned Mapping with Upright Display, \n"
    "2: Prior-Orthogonal Mapping with Upright Display, \n"
    "3: Prior-Aligned Mapping with Flat Display, \n"
    "4: Prior-Orthogonal Mapping with Flat Display"
    )

10. Run the **Default Settings** cell. You can easily run each cell by holding *shift* and pressing *return* on the keyboard.  
11. Run each block one by one, in order, by running the corresponding cell. A "completed" message will print and a bell sound will play when the block has completed. Give a few minutes of break between each block before prompting the participant on whether they are ready to continue, and running the next cell.

## Dependencies

In [ ]:
from riglib.experiment.scripting import run_experiment
import aopy
from aopy.data import db
from IPython.display import HTML
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import seaborn as sns
import os
import time

os.environ["BMI3D_PORT"] = "8000"

## Functions

In [ ]:
# ===========================================================================
# Plotting functions
# ===========================================================================

def plot_plane(plane, gain=1.0, color='grey', alpha=0.15, resolution=100,
               bounds=None, ax=None, **kwargs):

    if ax is None:
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')

    grid_range = np.linspace(-10*gain, 10*gain, resolution)

    if plane.shape in [(3, 3), (4, 4)]:
        x, y = np.meshgrid(grid_range, grid_range)
        coords = np.column_stack((x.ravel(), y.ravel(), np.zeros_like(x.ravel())))
        rotated_coords = coords @ plane[:3, :3]
        x, y, z = rotated_coords.T.reshape(3, *x.shape)

    elif plane.shape == (4,):
        A, B, C, D = plane
        # Solve for whichever axis has the largest |normal component| so a
        # near-vertical plane (|C| ≈ 0) doesn't blow up the surface.
        dep = int(np.argmax(np.abs([A, B, C])))
        if dep == 2:
            x, y = np.meshgrid(grid_range, grid_range)
            z = (-A*x - B*y - D) / C
        elif dep == 1:
            x, z = np.meshgrid(grid_range, grid_range)
            y = (-A*x - C*z - D) / B
        else:
            y, z = np.meshgrid(grid_range, grid_range)
            x = (-B*y - C*z - D) / A
    else:
        raise ValueError("Plane must be (3,3), (4,4), or (4,).")

    # Apply bounds clipping
    if bounds is not None:
        xmin, xmax, ymin, ymax, zmin, zmax = bounds
        mask = (
            (x >= xmin) & (x <= xmax) &
            (y >= ymin) & (y <= ymax) &
            (z >= zmin) & (z <= zmax)
        )
        x = np.where(mask, x, np.nan)
        y = np.where(mask, y, np.nan)
        z = np.where(mask, z, np.nan)

    ax.plot_surface(x, y, z, alpha=alpha, color=color, **kwargs)

    if bounds is not None:
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_zlim(zmin, zmax)

    return ax

def color_targets(target_locations, target_idx, colors, target_radius, alpha=1, transform_matrix=None, bounds=None, ax=None, **kwargs):
    assert len(target_locations) == len(target_idx), "Locations must be the same length as indices"
    
    target_locations = np.array(target_locations)
    target_idx = np.array(target_idx)
    
    unique_targets = np.unique(np.column_stack((target_idx, target_locations)), axis=0)
    assert len(colors) >= len(np.unique(target_idx)), "Not enough colors for unique indices"

    if ax is None:
        fig = plt.figure(figsize=(8, 8))
        ax = fig.add_subplot(111, projection='3d')

    for row in unique_targets:
        idx = int(row[0])
        loc = row[1:]
        plot_circles([loc], target_radius, colors[idx], alpha=alpha, transform_matrix=transform_matrix, bounds=bounds, ax=ax, **kwargs)

    ax.set_box_aspect([1, 1, 1])
    
def plot_circles(circle_positions, circle_radius, circle_color='b', transform_matrix=None, bounds=None, alpha=1, ax=None, **kwargs):
    if ax is None:
        ax = plt.gca(projection='3d')

    # Generate a 2D circle in the XY plane
    theta = np.linspace(0, 2 * np.pi, 100)
    x_circle = circle_radius * np.cos(theta)
    y_circle = circle_radius * np.sin(theta)
    z_circle = np.zeros_like(x_circle)  # Initially in the xy plane

    # Stack into a (3, N) array for matrix multiplication
    circle_points = np.vstack((x_circle, y_circle, z_circle))

    # Apply transformation matrix
    if transform_matrix is None:
        transform_matrix = np.eye(3)  # Default to xy plane

    transformed_points = transform_matrix @ circle_points  # Apply transformation

    for pos in circle_positions:
        pos = np.array(pos)

        # Apply translation
        x, y, z = transformed_points + pos[:, np.newaxis]

        # Create a filled polygon (triangle mesh) for the circle
        verts = [list(zip(x, y, z))]  # Format required for Poly3DCollection

        # Add the filled circle to the plot
        poly = Poly3DCollection(verts, color=circle_color, alpha=alpha, **kwargs)
        ax.add_collection3d(poly)

    if bounds is not None:
        set_bounds(bounds, ax)

In [ ]:
# ===========================================================================
# Plane finding functions
# ===========================================================================


# --- constants ---------------------------------------------------------------

_AXES = {'x': np.array([1., 0., 0.]),
         'y': np.array([0., 1., 0.]),
         'z': np.array([0., 0., 1.])}


NEW_DEG = 60


_S_DESIGN_TO_BMI3D = np.array([[1., 0., 0.], [0., 0., 1.], [0., 1., 0.]])


# --- rotations & geometry ----------------------------------------------------

def _shortest_rotation(d, n):
    """Shortest rotation R with R @ d = n. d, n must be unit. Raises if
    d ≈ -n (rotation axis ill-defined)."""
    cos_a = float(np.dot(d, n))
    if cos_a > 0.9999:
        return np.eye(3)
    if cos_a < -0.9999:
        axis = np.cross(d, np.array([1.0, 0.0, 0.0]))
        if np.linalg.norm(axis) < 1e-8:
            axis = np.cross(d, np.array([0.0, 1.0, 0.0]))
        axis = axis / np.linalg.norm(axis)
        K = np.array([[0, -axis[2], axis[1]],
                      [axis[2], 0, -axis[0]],
                      [-axis[1], axis[0], 0]])
        return np.eye(3) + 2.0 * (K @ K)
    axis = np.cross(d, n)
    sin_a = np.linalg.norm(axis)
    axis = axis / sin_a
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    return np.eye(3) + sin_a * K + (1 - cos_a) * (K @ K)


def _rot_axis(axis, angle):
    """Right-handed rotation by angle (rad) about an arbitrary axis (Rodrigues)."""
    axis = np.asarray(axis, float); axis = axis / np.linalg.norm(axis)
    c, s = np.cos(angle), np.sin(angle)
    K = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
    return np.eye(3) + s * K + (1 - c) * (K @ K)


def _display_normal(display):
    return np.array([0., 0., 1.]) if display == 'upright' else np.array([0., 1., 0.])


def _alt_axis(display):
    """Alternative (perpendicular) perturbation axis: y for upright, z for flat."""
    return 'y' if display == 'upright' else 'z'


def _ang(a, b):
    """Acute angle (deg) between two vectors."""
    a = np.asarray(a, float) / np.linalg.norm(a)
    b = np.asarray(b, float) / np.linalg.norm(b)
    return float(np.degrees(np.arccos(np.clip(abs(a @ b), 0.0, 1.0))))


# --- PCA plane fitting -------------------------------------------------------

def fit_PCA_plane(hand_traj):
    endpoints = np.vstack([traj[-1] for traj in hand_traj])
    origin = np.mean(endpoints, axis=0)
    centered = endpoints - origin

    # Perform PCA using SVD
    U, S, Vt = np.linalg.svd(centered)
    normal = Vt[-1]  # Normal vector is the last principal component (smallest variance direction)
    normal = normal / np.linalg.norm(normal)

    # Compute plane offset D such that the plane goes through the origin
    D = -np.dot(normal, origin)
    equation = np.append(normal, D)

    # Compute variance explained
    projected = centered @ normal
    var_total = np.sum(np.var(centered, axis=0))
    var_normal = np.var(projected)
    percent_explained = (1 - (var_normal/var_total)) if (var_total > 0) else 0

    # Standardize normal direction so it points into the +y/+z half-space
    # the half containing the mapping plane normals for both upright (+z) and
    # flat (+y) displays plus their moderate perturbations.
    if normal[1] + normal[2] < 0:
        normal = -normal
        equation[:3] = -equation[:3]
        equation[3] = -equation[3]

    return equation, origin, normal, percent_explained, var_normal


def align_PC_plane(n_PC, hand_traj, target_idx, display_targets, display_normal):
    """Align canonical display targets to subject reach data on the PC plane.

    Two steps:
      1. Pick the sign of n_PC by SSD between
         _shortest_rotation(display_normal, ±n_PC) @ canonical_targets and
         the reach centroids projected onto the PC plane. The chosen
         rotation is R_oop ("out of plane").
      2. Compute the in-plane rotation R_ip about the signed n_PC that
         minimizes SSD between R_ip @ R_oop @ canonical_targets and the
         projected centroids (closed-form 2D Procrustes).

    Returns (n_PC_signed, R_oop, R_ip).
    """
    n_PC = np.asarray(n_PC, dtype=float)
    target_idx = np.asarray(target_idx)
    endpoints = np.vstack([traj[-1] for traj in hand_traj])
    origin = endpoints.mean(axis=0)

    unique_idx = np.unique(target_idx)
    centroids = np.array([endpoints[target_idx == idx].mean(axis=0)
                          for idx in unique_idx])
    centroids_centered = centroids - origin
    projected = centroids_centered - (centroids_centered @ n_PC)[:, None] * n_PC

    canonical = np.asarray(display_targets)[unique_idx]

    # Step 1: pick sign and R_oop by SSD
    best_err = np.inf
    best_sign, R_oop = +1, None
    for s in (+1, -1):
        R = _shortest_rotation(display_normal, s * n_PC)
        rotated = canonical @ R.T
        err = float(np.sum((rotated - projected) ** 2))
        if err < best_err:
            best_err, best_sign, R_oop = err, s, R
    n_PC_signed = best_sign * n_PC

    # Step 2: in-plane rotation R_ip about n_PC_signed
    rotated_canonical = canonical @ R_oop.T
    crosses = np.cross(rotated_canonical, projected)
    sum_cross_n = float(crosses.sum(axis=0) @ n_PC_signed)
    sum_dot = float(np.sum(rotated_canonical * projected))
    phi = np.arctan2(sum_cross_n, sum_dot)

    K = np.array([[0, -n_PC_signed[2], n_PC_signed[1]],
                  [n_PC_signed[2], 0, -n_PC_signed[0]],
                  [-n_PC_signed[1], n_PC_signed[0], 0]])
    R_ip = np.eye(3) + np.sin(phi) * K + (1 - np.cos(phi)) * (K @ K)

    return n_PC_signed, R_oop, R_ip


def _fit_signed_normal(display, targets, target_idx, hand_traj):
    dn = _display_normal(display)
    _, _, n_PC, _, _ = fit_PCA_plane(hand_traj)
    n_ref, _, _ = align_PC_plane(n_PC, hand_traj, target_idx, targets, dn)
    return n_ref / np.linalg.norm(n_ref), dn


# --- no-mirror compliance ----------------------------------------------------

def _targets_compliant(R, display, display_targets, tol=1e-9):
    """No-mirror check: place targets by rotation R; every target must keep the
    sign of both its in-plane coords (right = x; up = y upright / forward = z
    flat). Returns (ok, mirrored) with mirrored a list of (target_idx, axis)."""
    inplane = (0, 1) if display == 'upright' else (0, 2)
    tgt = np.asarray(display_targets, float)
    mirrored = []
    for i, t in enumerate(tgt):
        h = R @ t
        for ax in inplane:
            if abs(t[ax]) > tol and np.sign(h[ax]) != np.sign(t[ax]):
                mirrored.append((i, ax))
    return (len(mirrored) == 0), mirrored


def _no_mirror_gap_deg(n, display_normal, display, display_targets, step=0.5, max_deg=90.0):
    """Angular distance (deg) from normal n to the no-mirror compliant region:
    smallest rotation of n (about any axis) whose roll-free placement is both-axis
    compliant. 0.0 if already compliant; nan if none within max_deg."""
    if _targets_compliant(_shortest_rotation(display_normal, n), display, display_targets)[0]:
        return 0.0
    m = 48; k = np.arange(m) + 0.5
    pol = np.arccos(1 - 2 * k / m); azi = np.pi * (1 + 5 ** 0.5) * k
    axes = np.c_[np.cos(azi) * np.sin(pol), np.sin(azi) * np.sin(pol), np.cos(pol)]
    for deg in np.arange(step, max_deg + step, step):
        ang = np.radians(deg)
        for ax in axes:
            if _targets_compliant(_shortest_rotation(display_normal, _rot_axis(ax, ang) @ n),
                                  display, display_targets)[0]:
                return float(deg)
    return float('nan')


# --- BMI3D angle conversion --------------------------------------------------

def bmi3d_angles(R_py):
    """Design-frame rotation R_py -> BMI3D perturbation params (degrees):
    (rot_x, rot_y, rot_z) for (perturbation_rotation_x, pertubation_rotation [y],
    perturbation_rotation_z), so BMI3D's row-vector pipeline v @ Ry @ Rz @ Rx
    reproduces R_py. (S swaps design y<->z; then extract YZX Euler of R_bmi^T.)"""
    from scipy.spatial.transform import Rotation as _Rot
    R_bmi = _S_DESIGN_TO_BMI3D @ R_py @ _S_DESIGN_TO_BMI3D
    a_y, a_z, a_x = _Rot.from_matrix(R_bmi.T).as_euler('YZX', degrees=True)
    return a_x, a_y, a_z


# --- plane generation (initial & alt-axis) ------------------------------

def parallel_plane(display, targets, target_idx, hand_traj):
    """Dual-axis Prior-Aligned initial: roll-free placement onto the subject's PC
    plane (the designed plane overlaps the PC plane; no in-screen roll)."""
    n_ref, dn = _fit_signed_normal(display, targets, target_idx, hand_traj)
    R = _shortest_rotation(dn, n_ref)
    n_design = R @ dn
    ok = _targets_compliant(R, display, targets)[0]
    return dict(n_ref=n_ref, n_design=n_design, R=R, compliant=ok,
                gap_deg=0.0 if ok else _no_mirror_gap_deg(n_design, dn, display, targets))


def orthogonal_plane(display, targets, target_idx, hand_traj):
    """Prior-orthogonal initial: 90 deg about the relative-x axis, roll-free,
    in the mirror-free region. If neither direction complies, keep the
    least-violating one (smallest no-mirror gap)."""
    n_ref, dn = _fit_signed_normal(display, targets, target_idx, hand_traj)
    R_oop = _shortest_rotation(dn, n_ref)
    x_rel = R_oop @ np.array([1., 0., 0.])
    chosen = None
    for sgn in (1.0, -1.0):
        n_d = _rot_axis(x_rel, sgn * np.pi / 2) @ n_ref
        R_d = _shortest_rotation(dn, n_d)
        if _targets_compliant(R_d, display, targets)[0]:
            chosen = (n_d, R_d); break
    if chosen is None:
        cands = [(_no_mirror_gap_deg(_rot_axis(x_rel, s * np.pi / 2) @ n_ref, dn, display, targets), s)
                 for s in (1.0, -1.0)]
        s = min(cands, key=lambda c: (np.inf if np.isnan(c[0]) else c[0]))[1]
        n_design = _rot_axis(x_rel, s * np.pi / 2) @ n_ref
        R_design = _shortest_rotation(dn, n_design)
        ok = False
    else:
        n_design, R_design = chosen; ok = True
    return dict(n_ref=n_ref, n_design=n_design, R=R_design, compliant=ok,
                gap_deg=0.0 if ok else _no_mirror_gap_deg(n_design, dn, display, targets))


def alt_axis_plane(display, targets, target_idx, hand_traj):
    """Compound perturbation from the z2 PC plane: rotate 60 deg about the
    relative-alt axis (y upright / z flat), roll-free, in the mirror-free region.
    When both directions comply keep the one further from the display; if neither
    complies keep the least-violating one (smallest no-mirror gap)."""
    n_ref, dn = _fit_signed_normal(display, targets, target_idx, hand_traj)
    R_oop = _shortest_rotation(dn, n_ref)
    alt_rel = R_oop @ _AXES[_alt_axis(display)]
    best = None
    for sgn in (1.0, -1.0):
        n_d = _rot_axis(alt_rel, sgn * np.radians(NEW_DEG)) @ n_ref
        R_d = _shortest_rotation(dn, n_d)
        if _targets_compliant(R_d, display, targets)[0]:
            tilt = _ang(n_d, dn)
            if best is None or tilt > best[0]:
                best = (tilt, n_d, R_d)
    if best is None:
        cands = [(_no_mirror_gap_deg(_rot_axis(alt_rel, s * np.radians(NEW_DEG)) @ n_ref,
                                     dn, display, targets), s) for s in (1.0, -1.0)]
        s = min(cands, key=lambda c: (np.inf if np.isnan(c[0]) else c[0]))[1]
        n_design = _rot_axis(alt_rel, s * np.radians(NEW_DEG)) @ n_ref
        R_design = _shortest_rotation(dn, n_design)
        ok = False
    else:
        _, n_design, R_design = best; ok = True
    return dict(n_ref=n_ref, n_design=n_design, R=R_design, compliant=ok,
                gap_deg=0.0 if ok else _no_mirror_gap_deg(n_design, dn, display, targets))


## Defaults

In [ ]:
## ----- for subjects recorded in the past
# preproc_dir = '/media/moor-data/preprocessed-human'
# data_dir = '/data/human'

# ----- for subjects being recorded now
preproc_dir = '/home/aolab/code/notebooks/temp_data'
data_dir = '/data'

db.BMI3D_DBNAME = "human"
sources = ['exp']

upright_targets = np.array([
    [0., 0., 0.],
    [0., 10., 0.],
    [7.0711, 7.0711, 0.],
    [10., 0., 0.],
    [7.0711, -7.0711, 0.],
    [0., -10., 0.],
    [-7.0711, -7.0711, 0.],
    [-10., 0., 0.],
    [-7.0711, 7.0711, 0.]
])

flat_targets = np.array([t[[0,2,1]] for t in upright_targets])

display='upright' if (group==1 or group==2) else 'flat'

display_targets=upright_targets if display=='upright' else flat_targets

display_normal = np.array([0., 0., 1.]) if display == 'upright' else np.array([0., 1., 0.])

target_ids = np.arange(len(upright_targets))

target_colors = sns.color_palette(n_colors=len(target_ids))
target_colors[0] = (1.0, 1.0, 1.0, 0.0)

bounds = (-10, 10, -10, 10, -10, 10)
bounds_plus = (-11, 11, -11, 11, -11, 11)

In [ ]:
# Generate and save default settings

project = "DPS"
task = "manual control"

features = ["autostart",
     "optitrack",
     "penalty_audio",
     "rand_delay",
     "reward_audio",
     "saveHDF",
     "syncHDF",
     "windowVR"]

parameters = {
    "scale": 1,
    "offset": [-20, -95, -2],
    "reward_sound": "incorrect.wav",
    "penalty_sounds": "buzzer.wav",
    "rand_start": [0.25, 0.75],
    "rand_delay": [0,0],
    "prob_catch_trials": 0,
    "show_grid": True,
    "grid_size": 130,
    "fixed_camera_position": False,
    "fixed_camera_orientation": False,
    "fullscreen": True,
    "session_length": 0,
    "velocity_control": False,
    "random_rewards": False,
    "rotation": "yzx",
    "exp_rotation": "none",
    "pertubation_rotation": 0, # this is perturbation_rotation_y, misspelling is unfortunately system-side
    "perturbation_rotation_z": 0,
    "perturbation_rotation_x": 0,
    "exp_gain": 1.0,
    "limit2d": False,
    "target_radius": 1,
    "plant_type": "cursor",
    "plant_visible": True,
    "reward_time": 0.5,
    "hold_time": 0.2,
    "hold_penalty_time": 1,
    "delay_penalty_time": 1,
    "timeout_time": 10,
    "timeout_penalty_time": 1,
    "max_attempts": 3,
    "num_targets_per_attempt": 2
    }

if group==1 or group==2: # Upright display
    generator = "centerout_2d"
    parameters['cursor_bounds'] = [-14,14,0,0,-14,14]
    parameters['grid_position'] = [0, 0, 130-eye_height]
    parameters['camera_offset'] = [0, -eye_height, 50]
elif group==3 or group==4: # Flat display
    generator = "centerout_tabletop"
    parameters['cursor_bounds'] = [-14,14,-14,14,0,0]
    parameters['grid_position'] = [0, 0, 160-eye_height]
    parameters['camera_offset'] = [0, 30-eye_height, 50]

settings = {"features": features, "parameters": parameters}

with open('DPS_defaults.pkl', 'wb') as f:
    pickle.dump(settings, f)

def play_sound(file):
    return HTML(f"""
    <audio autoplay>
      <source src="{file}" type="audio/mpeg">
    </audio>
    """)

# Blind Block (z1)
* Remind participants that they will not see a cursor for this block, but to move their hand from the center to where they think the target is and hold there until the target disappears.

In [ ]:
# time.sleep(20) # wait 20 seconds before running

with open('DPS_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)
    
parameters = settings['parameters']
features = settings['features']

session = "z1"
nblocks = 4

task_desc = "Zero-feedback"

remove_feats = ['penalty_audio','reward_audio']
features = [feat for feat in features if feat not in remove_feats]

parameters['exp_gain'] = 0
parameters['plant_visible'] = False
parameters['timeout_time'] = 1.5
parameters['hold_time'] = 1.5
parameters['hold_penalty_time'] = 0
parameters['delay_penalty_time'] = 0
parameters['timeout_penalty_time'] = 0
parameters['rand_start'] = [0.25, 0.75]
parameters['max_attempts'] = 1

sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

print("Blind block (z1) completed.")
play_sound("Bell.wav")

## Calculate Initial Plane

In [ ]:
# identify entry id for the correct z1 block

# !/home/aolab/code/nightly_script.sh

entries = db.lookup_mc_sessions(subject=subject, session='z1')

db.summarize_entries(entries, sum_trials=False)

* Manually specify the te_id for the z1 block

In [ ]:
z1_id = 6167





# --- Preprocess and tabulate data --------------------

entry = db.lookup_mc_sessions(id=z1_id)

subject_id, te_id, date = db.list_entry_details(entry)
files = entry[0].get_raw_files()

preproc_dir_subj = os.path.join(preproc_dir, subject)

aopy.preproc.proc_single(data_dir, files, preproc_dir_subj, subject, te_id[0], date[0], sources, overwrite=True)

df = aopy.data.tabulate_behavior_data_center_out(preproc_dir, subject_id, te_id, date, 
        metadata=['session', 'target_radius', 'exp_rotation', 'scale', 'rand_delay'])

hand_traj = aopy.data.bmi3d.tabulate_kinematic_data(
                preproc_dir,
                df['subject'],
                df['te_id'],
                df['date'],
                df['go_cue_time'],
                df['trial_end_time'],
                low_cut=12, buttord=2,
                datatype='user_world')

df['hand_traj'] = None
df['hand_traj'] = df['hand_traj'].astype(object)

for trial,seg in zip(df.index.tolist(), hand_traj):
    df.at[trial, 'hand_traj'] = seg - seg[0]

In [ ]:
# --- Calculate upcoming intial plane angles --------------------

hand_traj = list(df['hand_traj'].values)
target_idx = df['target_idx'].values

if group in [1, 3]:
    res = parallel_plane(display, display_targets, target_idx, hand_traj)
    kind = 'prior-aligned'
elif group in [2, 4]:
    res = orthogonal_plane(display, display_targets, target_idx, hand_traj)
    kind = 'prior-orthogonal'
else:
    raise ValueError(f"unexpected group {group!r}")

initial_Rx, initial_Ry, initial_Rz = bmi3d_angles(res['R'])
n_ref, n_design, R = res['n_ref'], res['n_design'], res['R']

print(f'Subject: {subject}, display: {display}, group: {group}  [{kind}]')
print(f'  BMI3D params (deg):  Rx = {initial_Rx:+.4f}   Ry = {initial_Ry:+.4f}   Rz = {initial_Rz:+.4f}')
if not res['compliant']:
    print('  ' + '!' * 64)
    print(f"  WARNING: no mirror-free initial plane found; most-compliant plane used")
    print(f"           (no-mirror: {res['gap_deg']:.1f} deg outside the compliant region).")
    print('  ' + '!' * 64)

    
    
# --- Visualize blind and upcoming initial plane --------------------

endpoints = np.vstack([traj[-1] for traj in hand_traj])
endpoints_centered = endpoints - endpoints.mean(axis=0)
ep_colors = [target_colors[int(t)] for t in target_idx]

R_pc_pos  = _shortest_rotation(display_normal, n_ref)
R_pc_disk = _shortest_rotation(np.array([0., 0., 1.]), n_ref)
R_des_disk = _shortest_rotation(np.array([0., 0., 1.]), n_design)

NORMAL_LEN = 8
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': '3d'})
ax.set_title(f'{subject} ({display}) -- {kind}', fontsize=11)

plot_plane(np.append(n_ref, 0.0), gain=1.0, color='green', alpha=0.15, bounds=bounds, ax=ax)
color_targets(display_targets @ R_pc_pos.T, target_ids, target_colors,
              target_radius=2, alpha=0.08, bounds=None, ax=ax,
              transform_matrix=R_pc_disk, linewidth=0.4)
ax.quiver(0, 0, 0, n_ref[0], n_ref[1], n_ref[2], length=NORMAL_LEN, color='green',
          linewidth=3, arrow_length_ratio=0.18, normalize=True)

plot_plane(np.append(n_design, 0.0), gain=1.0, color='royalblue', alpha=0.15, bounds=bounds, ax=ax)
color_targets(display_targets @ R.T, target_ids, target_colors,
              target_radius=2, alpha=0.45, bounds=None, ax=ax,
              transform_matrix=R_des_disk, linewidth=1.5)
ax.quiver(0, 0, 0, n_design[0], n_design[1], n_design[2], length=NORMAL_LEN,
          color='royalblue', linewidth=3, arrow_length_ratio=0.18, normalize=True)

ax.scatter(endpoints_centered[:, 0], endpoints_centered[:, 1],
           endpoints_centered[:, 2], c=ep_colors, alpha=0.5, s=20)
ax.set_xlabel('x', labelpad=5); ax.set_ylabel('y', labelpad=5); ax.set_zlabel('z', labelpad=5)
ax.set_xlim(bounds[0], bounds[1]); ax.set_ylim(bounds[2], bounds[3]); ax.set_zlim(bounds[4], bounds[5])
ax.view_init(elev=15, azim=180 + 60, roll=0, vertical_axis='y')
plt.tight_layout()

# Initial Session
* Let participants know that the cursor will now be controlled with their hand motions.
* Don't give further details on the control mapping. If they ask, just say: "That's not something that I can tell you."

In [ ]:
# Block 1 (a1)

with open('DPS_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)
    
parameters = settings['parameters']
features = settings['features']

parameters['perturbation_rotation_x'] = initial_Rx
parameters['pertubation_rotation'] = initial_Ry
parameters['perturbation_rotation_z'] = initial_Rz
    
session = "a1"
nblocks = 8

if group==1 or group==3: # prior-aligned
    task_desc = "Prior-aligned initial mapping"
elif group==2 or group==4: # prior-orthogonal
    task_desc = "Prior-orthogonal initial mapping"
    
    
sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

print("First initial block (a1) completed.")
play_sound("Bell.wav")

In [ ]:
# Block 2 (a2)

session = "a2"

run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

print("Second initial block (a2) completed.")
play_sound("Bell.wav")

In [ ]:
# Block 3 (a3)

session = "a3"

run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

print("Third initial block (a3) completed.")
play_sound("Bell.wav")

# Catch Block (z2)
* Let people know that this will be another block where they can't see the cursor, but to once again move their hand from the center to where they think the target is and hold until the target disappears, then return to center.

In [ ]:
with open('DPS_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)
    
parameters = settings['parameters']
features = settings['features']

session = "z2"
nblocks = 4

task_desc = "Zero-feedback"

remove_feats = ['penalty_audio','reward_audio']
features = [feat for feat in features if feat not in remove_feats]

parameters['exp_gain'] = 0
parameters['plant_visible'] = False
parameters['timeout_time'] = 1.5
parameters['hold_time'] = 1.5
parameters['hold_penalty_time'] = 0
parameters['delay_penalty_time'] = 0
parameters['timeout_penalty_time'] = 0
parameters['rand_start'] = [0.25, 0.75]
parameters['max_attempts'] = 1

sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

print("Catch block (z2) completed.")
play_sound("Bell.wav")

## Calculate Alt-Axis Plane

In [ ]:
# identify entry id for the correct z1 block

entries = db.lookup_mc_sessions(subject=subject, session='z2')

db.summarize_entries(entries, sum_trials=False)

* Manually identify te_id for the z2 block

In [ ]:
z2_id = 4537





# --- Preprocess and tabulate data --------------------

entry = db.lookup_mc_sessions(id=z2_id)

subject_id, te_id, date = db.list_entry_details(entry)
files = entry[0].get_raw_files()

preproc_dir_subj = os.path.join(preproc_dir, subject)

aopy.preproc.proc_single(data_dir, files, preproc_dir_subj, subject, te_id, date, sources, overwrite=True)

df = aopy.data.tabulate_behavior_data_center_out(preproc_dir, subject_id, te_id, date, 
        metadata=['session', 'target_radius', 'exp_rotation', 'scale', 'rand_delay'])

hand_traj = aopy.data.bmi3d.tabulate_kinematic_data(
                preproc_dir,
                df['subject'],
                df['te_id'],
                df['date'],
                df['go_cue_time'],
                df['trial_end_time'],
                low_cut=12, buttord=2,
                datatype='user_world')

df['hand_traj'] = None
df['hand_traj'] = df['hand_traj'].astype(object)

for trial,seg in zip(df.index.tolist(), hand_traj):
    df.at[trial, 'hand_traj'] = seg - seg[0]

In [ ]:
# --- Calculate upcoming alt_axis plane angles --------------------

hand_traj = list(df['hand_traj'].values)
target_idx = df['target_idx'].values

# from the z2 PC plane, rotate 60 deg about the relative-alt axis
# (y upright / z flat), roll-free, into the both-axis compliant region (further
# from the display when both comply; least-violating if neither).
res = alt_axis_plane(display, display_targets, target_idx, hand_traj)
alt_axis_Rx, alt_axis_Ry, alt_axis_Rz = bmi3d_angles(res['R'])
n_z2, n_compound, R = res['n_ref'], res['n_design'], res['R']

print(f'Subject: {subject}, display: {display}, group: {group}')
print(f'  BMI3D params (deg):  Rx = {alt_axis_Rx:+.4f}   Ry = {alt_axis_Ry:+.4f}   Rz = {alt_axis_Rz:+.4f}')
if not res['compliant']:
    print('  ' + '!' * 64)
    print(f"  WARNING: no mirror-free compound plane found; most-compliant plane used")
    print(f"           (no-mirror: {res['gap_deg']:.1f} deg outside the compliant region).")
    print('  ' + '!' * 64)

# --- Visualize blind and upcoming alt_axis plane --------------------
endpoints = np.vstack([traj[-1] for traj in hand_traj])
endpoints_centered = endpoints - endpoints.mean(axis=0)
ep_colors = [target_colors[int(t)] for t in target_idx]

R_z2_pos = _shortest_rotation(display_normal, n_z2)
R_z2_disk = _shortest_rotation(np.array([0., 0., 1.]), n_z2)
R_cmp_disk = _shortest_rotation(np.array([0., 0., 1.]), n_compound)

NORMAL_LEN = 8
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': '3d'})
ax.set_title(f'{subject} ({display}) -- alt-axis plane', fontsize=11)

plot_plane(np.append(n_z2, 0.0), gain=1.0, color='green', alpha=0.15, bounds=bounds, ax=ax)
color_targets(display_targets @ R_z2_pos.T, target_ids, target_colors,
              target_radius=2, alpha=0.08, bounds=None, ax=ax,
              transform_matrix=R_z2_disk, linewidth=0.4)
ax.quiver(0, 0, 0, n_z2[0], n_z2[1], n_z2[2], length=NORMAL_LEN, color='green',
          linewidth=3, arrow_length_ratio=0.18, normalize=True)

plot_plane(np.append(n_compound, 0.0), gain=1.0, color='royalblue', alpha=0.15, bounds=bounds, ax=ax)
color_targets(display_targets @ R.T, target_ids, target_colors,
              target_radius=2, alpha=0.45, bounds=None, ax=ax,
              transform_matrix=R_cmp_disk, linewidth=1.5)
ax.quiver(0, 0, 0, n_compound[0], n_compound[1], n_compound[2], length=NORMAL_LEN,
          color='royalblue', linewidth=3, arrow_length_ratio=0.18, normalize=True)

ax.scatter(endpoints_centered[:, 0], endpoints_centered[:, 1],
           endpoints_centered[:, 2], c=ep_colors, alpha=0.5, s=20)
ax.set_xlabel('x', labelpad=5); ax.set_ylabel('y', labelpad=5); ax.set_zlabel('z', labelpad=5)
ax.set_xlim(bounds[0], bounds[1]); ax.set_ylim(bounds[2], bounds[3]); ax.set_zlim(bounds[4], bounds[5])
ax.view_init(elev=15, azim=180 + 60, roll=0, vertical_axis='y')
plt.tight_layout()


# Alt-Axis Session
* Let them know they'll be controlling the cursor again. Don't say anything about the mapping, including confirming to them that it's changed. If asked, say "That's not something that I can tell you."

In [ ]:
# Block 1 (b1)

with open('DPS_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)
    
parameters = settings['parameters']
features = settings['features']

parameters['pertubation_rotation'] = alt_axis_Ry
parameters['perturbation_rotation_z'] = alt_axis_Rz
parameters['perturbation_rotation_x'] = alt_axis_Rx
    
session = "b1"
nblocks = 6

task_desc = "Dynamic Alternative-Axis Mapping"
    
sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

print("First alt-axis block (b1) completed.")
play_sound("Bell.wav")

In [ ]:
# Block 2 (b2)
    
session = "b2"
    
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

print("Second alt-axis block (b2) completed.")
play_sound("Bell.wav")

In [ ]:
# Block 3 (b3)
    
session = "b3"
    
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

print("Third alt-axis block (b3) completed.")
play_sound("Bell.wav")

In [ ]:
# Block 4 (b4)
    
session = "b4"
    
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

print("Fourth alt-axis block (b4) completed.")
play_sound("Bell.wav")

# Washout Session
* Don't tell them anything will change between the last block and this one. Say "That's not something I can tell you" if they ask if it changed, or stay silent if they make a comment.

In [ ]:
# Block 1 (aprime1)

with open('DPS_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)

parameters = settings['parameters']
features = settings['features']

# Washout returns to the initial (A) mapping -> reuse the initial plane.
parameters['perturbation_rotation_x'] = initial_Rx
parameters['pertubation_rotation'] = initial_Ry
parameters['perturbation_rotation_z'] = initial_Rz

session = "aprime1"
nblocks = 8

if group == 1 or group == 3:      # prior-aligned
    task_desc = "Prior-aligned washout"
elif group == 2 or group == 4:    # prior-orthogonal
    task_desc = "Prior-aligned washout"

sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject, experimenter, project, session, task, features, sequence, task_desc, save, **parameters)

print("First washout block (aprime1) completed.")
play_sound("Bell.wav")


In [ ]:
# Block 2 (aprime2)

session = "aprime2"
    
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

print("Second washout block (aprime2) completed.")
play_sound("Bell.wav")

print("Don't forget to put files on moor!")